# Lab 1a: Linear Regression with scikit-learn

## Dataset: Student Performance

This is the "easy" companion to `linear_regression_scratch.ipynb`. Same problem, same
dataset - but instead of writing your own cost function and gradient descent, you will use
**scikit-learn**'s built-in `LinearRegression`, which handles the training for you.

**Goal:** predict a student's `Final_Score` from `Study_Hours`, `Attendance` and
`Practice_Tests`.

## Notebooks in this lab
| | from scratch | scikit-learn |
|---|---|---|
| **Linear Regression** | `linear_regression_scratch.ipynb` | **`linear_regression_sklearn.ipynb`** (this one) |
| **Logistic Regression** | `logistic_regression_scratch.ipynb` | `logistic_regression_sklearn.ipynb` |

## Outline
- [1 - Packages](#1)
- [2 - Load the Dataset](#2)
- [3 - Explore & Prepare the Data](#3)
  - [3.1 Data types](#3.1)
  - [3.2 Train / Test split](#3.2)
- [4 - Train the model](#4)
- [5 - Predict & evaluate](#5)

<a name="1"></a>
## 1 - Packages

- [numpy](https://www.numpy.org) for array math
- [pandas](https://pandas.pydata.org) to load and manipulate the dataset
- [matplotlib](https://matplotlib.org) to plot the results
- [scikit-learn](https://scikit-learn.org) for the train/test split and the model itself

In [ ]:
# On Kaggle / Colab: get the lab files. Skip this cell if you already run the notebook
# from inside the repository folder.
!git clone https://github.com/MLs-labs/Lab_1_ML

In [ ]:
import os
import sys

# Works both on Kaggle (after the git clone above) and locally from the repo folder.
REPO_DIR = "/kaggle/working/Lab_1_ML"
if not os.path.isdir(REPO_DIR):
    REPO_DIR = "."
sys.path.append(REPO_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# sklearn_lab_tests.py contains the checks used throughout this notebook
from sklearn_lab_tests import *

<a name="2"></a>
## 2 - Load the Dataset

The dataset `studentPerformance.csv` contains, for each student:
- `Study_Hours`: number of hours studied per day
- `Attendance`: attendance percentage
- `Practice_Tests`: number of practice tests taken
- `Final_Score`: final exam score (0-100) &rarr; **target for Linear Regression**
- `Pass_Fail`: 1 if the student passed, 0 otherwise &rarr; **target for Logistic Regression**

In [ ]:
df = pd.read_csv(os.path.join(REPO_DIR, "studentPerformance.csv"))

# Always look at your data before doing anything else.
print("Shape of the dataset (rows, columns):", df.shape)
df.head()

<a name="3"></a>
## 3 - Explore & Prepare the Data

<a name="3.1"></a>
### 3.1 Data types

Before building a model, you need to know what you are working with. `df.dtypes` tells you
the type pandas inferred for every column (`int64`, `float64`, `object`, ...). This matters
because a model only understands numbers, so any non-numeric column would need to be
encoded first.

In [ ]:
print(df.dtypes)

# `df.info()` gives a more complete summary: dtypes, non-null counts and memory usage.
df.info()

# Check for missing values.
print("\nMissing values per column:\n", df.isnull().sum())

# Quick statistical summary (mean, std, min, max, quartiles) for every numeric column.
df.describe()

All 5 columns are numeric and there are no missing values, so the dataset is usable as-is.

<a name="3.2"></a>
### 3.2 Train / Test split

We split off the input features `X` and the two targets (`y_reg` for the regression task,
`y_clf` for the classification task), then use scikit-learn's
[`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
to randomly split all three arrays into a training set and a test set - splitting them
together keeps a given student's row aligned across `X`, `y_reg` and `y_clf`, and gives you
the exact same split in every notebook of this lab.

In [ ]:
feature_cols = ["Study_Hours", "Attendance", "Practice_Tests"]

X = df[feature_cols].to_numpy()
y_reg = df["Final_Score"].to_numpy()
y_clf = df["Pass_Fail"].to_numpy()

### START CODE HERE ### (~ 1 line of code)
# split X, y_reg and y_clf together with test_size=0.2 and random_state=1
X_train, X_test, y_train_reg, y_test_reg, y_train_clf, y_test_clf = None, None, None, None, None, None
### END CODE HERE ###

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)

Run the cell below to check that your split is correct.

In [ ]:
split_test(X_train, X_test, y_train_reg, y_test_reg, y_train_clf, y_test_clf)

<a name="4"></a>
## 4 - Train the model

Create a [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
model and fit it on the training data with `.fit(X, y)`. `.fit()` returns the model itself,
so you can create and fit it in a single line.

In [ ]:
### START CODE HERE ### (~ 1 line of code)
lin_reg = None
### END CODE HERE ###

Run the cell below to check your trained model.

In [ ]:
linear_model_test(lin_reg)

The fitted parameters are stored on the model: one coefficient per feature in `coef_`, plus
the intercept in `intercept_`.

In [ ]:
for name, coef in zip(feature_cols, lin_reg.coef_):
    print(f"{name:>15}: {coef: .4f}")
print(f"{'intercept':>15}: {lin_reg.intercept_: .4f}")

<a name="5"></a>
## 5 - Predict & evaluate

Use the trained model to predict on the test set, then compare the predictions to the
actual scores with two standard regression metrics:
- **MSE** (mean squared error): average squared difference between predicted and actual
  scores - lower is better.
- **R²**: fraction of the variance in the scores explained by the model - closer to 1 is
  better.

In [ ]:
y_pred_reg = lin_reg.predict(X_test)

mse = mean_squared_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

print(f"MSE on the test set: {mse:.2f}")
print(f"R² on the test set:  {r2:.3f}")

#### Predicted vs. actual scores

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test_reg, y_pred_reg, alpha=0.6)
lims = [min(y_test_reg.min(), y_pred_reg.min()), max(y_test_reg.max(), y_pred_reg.max())]
plt.plot(lims, lims, c="r", linestyle="--", label="perfect prediction")
plt.title("Linear Regression - Predicted vs. Actual Final Score")
plt.xlabel("Actual Final Score")
plt.ylabel("Predicted Final Score")
plt.legend()
plt.show()

**Congratulations!** You trained a Linear Regression model with scikit-learn: splitting the
data, fitting the model in a single line, and evaluating it with standard regression
metrics.

Next: open `linear_regression_scratch.ipynb` to see everything scikit-learn was doing for
you under the hood, or `logistic_regression_sklearn.ipynb` for the classification task on
the same dataset.